# HachimiMT — Benchmark 1× T4 vs 2× T4 (Kaggle)

CTranslate2 hỗ trợ multi-GPU qua `device_index=[0,1]`. App hiện chỉ dùng 1 GPU.
Notebook này đo **dùng 2 T4 có nhanh hơn 1 T4 bao nhiêu** trên cùng văn bản, để
quyết định có đáng thêm multi-GPU vào app không.

**Cách dùng:** `Settings → Accelerator → GPU T4 x2` + `Internet → on`, rồi `Run All`.
Cell cuối in bảng — **báo lại** để chốt.

In [ ]:
# 1. Tải code + cài thư viện
import urllib.request, zipfile, os, shutil
ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)
with zipfile.ZipFile("hachimimt-local.zip") as z: z.extractall(".")
!pip install -q -r hachimimt/requirements.txt
import ctranslate2
print("CT2 cuda devices:", ctranslate2.get_cuda_device_count(), "(cần = 2 cho test này)")

In [ ]:
# 2. Chuẩn bị model + tokenizer + văn bản test (nhiều câu đa dạng)
import os, sys, time
sys.path.insert(0, os.path.abspath("hachimimt/src"))
from translator import HachimiTranslator, MODELS, Backend, ensure_model_files
from token_chunker import split_for_translation
from hardware import detect_hardware_profile

cfg = MODELS["HachimiMT-60"]
model_path = ensure_model_files(cfg, Backend.CT2)
ct2_dir = str(model_path / cfg.ct2_subdir)

# Dùng translator của app để có tokenizer + hàm token hóa đúng (chạy nhanh, chỉ để tokenize).
_app_tr = HachimiTranslator(detect_hardware_profile())
_app_tr.load("HachimiMT-60", Backend.CT2)

PARA = (
    "他抬头看向远处的山门，心中升起一股莫名的悸动。"
    "师兄说得对，我等修士当以大道为重，岂能因私情误了修行。"
    "她转身离去，没有再回头看一眼，仿佛一切已成定局。"
    "凌伊山掏出手机，查询起了最近开往雪霏市的机票。"
    "玄中门欲炼的丹药乃是凝婴丹，主材之一便是天婴果。"
    "他必须得抓紧时间了，否则一切都将来不及挽回。"
)
text = PARA * 600       # nhiều nghìn câu — đủ lớn để thấy chênh lệch GPU
chunks = split_for_translation(_app_tr._tokenizer, text, max_tokens=256, chunk_mode="sentence")
# token hóa sẵn (loại overhead tokenize khỏi phép đo GPU thuần)
tokenized = _app_tr._source_tokens_batch(chunks)
print(f"Văn bản: {len(text):,} ký tự → {len(chunks):,} chunk")

In [ ]:
# 3. Benchmark MA TRẬN: tách scaling GPU thuần vs thêm worker/GPU
#    inter_threads = số REPLICA trên MỖI GPU (CT2: num_replicas_per_device).
#    → [0,1]+inter=2 = 4 worker (2/GPU), KHÔNG phải "2 GPU". Để đo GPU thuần phải
#      so 1GPU·1worker vs 2GPU·1worker. (Phát hiện từ review ChatGPT.)
import gc, hashlib, statistics, time

assert ctranslate2.get_cuda_device_count() >= 2, "Cần ≥2 GPU CUDA (chọn T4 x2)"

# Giải phóng model tokenize khỏi VRAM (tránh lệch VRAM giữa GPU / OOM khi nhiều replica).
try:
    del _app_tr
except NameError:
    pass
gc.collect()

import subprocess
print("CTranslate2:", ctranslate2.__version__)
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

MAX_BATCH = 96 * cfg.ct2_max_input_tokens   # 24576 token — giống app (đừng để 96!)
OPTS = dict(max_batch_size=MAX_BATCH, batch_type="tokens", beam_size=1,
            no_repeat_ngram_size=2, repetition_penalty=1.2)
SRC_TOK = sum(len(t) for t in tokenized)
WARM = tokenized[:min(512, len(tokenized))]

def fingerprint(results):
    d = hashlib.sha256()
    for r in results:
        d.update(("\0".join(r.hypotheses[0])).encode("utf-8")); d.update(b"\n")
    return d.hexdigest()[:16]

def bench(device_index, label, inter_threads=1, reps=5):
    tr = ctranslate2.Translator(ct2_dir, device="cuda", device_index=device_index,
                                compute_type="int8_float16", inter_threads=inter_threads)
    for _ in range(2):
        tr.translate_batch(WARM, **OPTS)   # warmup cả 2 GPU
    times, ref = [], None
    for _ in range(reps):
        t0 = time.perf_counter()
        res = tr.translate_batch(tokenized, **OPTS)
        times.append(time.perf_counter() - t0)
        h = fingerprint(res)
        ref = ref or h
        assert h == ref, "Output KHÔNG ổn định giữa các lần chạy!"
        del res
    med = statistics.median(times)
    print(f"  {label:24} workers={len(device_index)*inter_threads} · {med:.2f}s · "
          f"{SRC_TOK/med:,.0f} src tok/s · hash={ref}")
    del tr; gc.collect()
    return {"label": label, "ndev": len(device_index), "tok_s": SRC_TOK/med, "hash": ref}

print(f"\nDịch {len(chunks):,} chunk ({SRC_TOK:,} src token), max_batch={MAX_BATCH}:")
rows = [
    bench([0],    "1×T4 · 1 worker/GPU", 1),
    bench([0, 1], "2×T4 · 1 worker/GPU", 1),
    bench([0],    "1×T4 · 2 worker/GPU", 2),
    bench([0, 1], "2×T4 · 2 worker/GPU", 2),
]

# Mọi cấu hình greedy phải cho CÙNG output.
assert len({r["hash"] for r in rows}) == 1, "⚠️ Output KHÁC nhau giữa cấu hình GPU!"

by = {r["label"]: r for r in rows}
pure = by["2×T4 · 1 worker/GPU"]["tok_s"] / by["1×T4 · 1 worker/GPU"]["tok_s"]
best1 = max(r["tok_s"] for r in rows if r["ndev"] == 1)
best2 = max(r["tok_s"] for r in rows if r["ndev"] == 2)
print(f"\n>>> GPU scaling THUẦN (2GPU·1w / 1GPU·1w): {pure:.2f}×")
print(f">>> Best 2 GPU / Best 1 GPU:             {best2/best1:.2f}×")
print("    (src token/s đáng tin hơn chunk/s · báo lại 2 số này để chốt)")